### Dimensionality Reduction 
1. Mode Shape Approach
2. PCA

In [ ]:
#generating the stiffness matrix
#so we generate and store 'n' normal distribution values 
#then create a matrix of the form of stiffness matrix, and put in the above values of k1-kn
import numpy as np
import matplotlib.pyplot as plt
import math

n = int(input("Enter no. of springs/masses(n): "))
sd = float(input("Enter standard deviation of normal distribution for stiffness matrix: "))
mean = 55

k = np.random.normal(loc=mean, scale=sd, size=n)

#so now we have k0 to kn-1...n values
plt.hist(k, bins=20)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Normal Distribution (Bell Curve)")
plt.show()

#similarly, we generate values of force
f = np.random.normal(loc=0, 2, size=n)

# Initialize stiffness matrix
K = np.zeros((n, n))

# Fill matrix
for i in range(n):
    # Diagonal terms
    if n == 1:
        K[i, i] = k[i]
    else: 
        if i == n - 1:
            K[i, i] = k[i]
        else:
            K[i, i] = k[i] + k[i+1]

    # non-diagonal terms
    if i < n - 1:
        K[i+1, i] = -k[i+1]
        K[i, i+1] = -k[i+1]

# frequency of certain values of K appearing is the normal distribution, the values themselves don't follow a bell curve


### MODE SHAPE APPROACH
1. find eigen values of the matrix K
2. express u in terms of the eigen vectors of K...u = f(1/a) where a is an eigenvector
3. then eliminate terms which tend to zero cause 'a' is too big

Mode Shape Approach uses eigen values of K

PCA Approach uses eigen values of covariance matrix of u to find direction of maximum variance

In [ ]:
ev_K, evec_K = np.linalg.eigh(K)  # ascending eigenvalues, columns = mode shapes

# Fix one force sample (so r is the only variable)
f = np.random.normal(0, 1, n)
u_exact = np.linalg.solve(K, f)  # ground truth

# Project force into modal coordinates
f_new = evec_K.T @ f         # shape (n,)

# Solve in modal space: q_i = f_modal_i / lambda_i
q_full = f_new / ev_K               # shape (n,)

# Sweep r from 1 to n
mse_mode_vs_r = []
r_list = list(range(1, n+1))

for r in r_list:
    # Reconstruct using only first r modes (lowest eigenvalues)
    q_reduced = np.zeros(n)
    q_reduced[:r] = q_full[:r]
    u_approx = evec_K @ q_reduced      # back to physical space
    
    mse = np.mean((u_exact - u_approx)**2)
    mse_mode_vs_r.append(mse)

plt.figure()
plt.semilogy(r_list, mse_mode_vs_r)
plt.xlabel("Number of modes retained (r)")
plt.ylabel("MSE")
plt.title("Mode Shape Approach: MSE vs r")
plt.grid(True)
plt.show()

### PCA Approach

for displacement u,
1. find mean
2. centre the data
3. compute covariance matrix
4. compute eigen values and eigenvectors
5. error = sum of eigenvalues from r+1 to d
6. compute fraction of variance from r = 1 to r=r .... = f(r)
7. choose smallest r such that f(r) >= threshold
8. give reduced basis from r = 1 to r = r

no. of displacements/masses/degrees of freedom = n

no. of samples = m

so our data matrix is of order n x m

stiffness matrix is same as above, for PCA we define the data matrix for u (n x m)



In [ ]:
#defining Data Matrix(U)
m = 100 #no. of samples
U = np.zeros((n, m))
sdev = 1

for j in range(m):
    f = np.random.normal(0, sdev, n)
    sdev += 1
    #f = np.random.randn(n)

# add correlation, to see if it's reducing dimensionality in any direction
   # f = f + 0.8*np.roll(f, 1)
    u = np.linalg.solve(K, f)           # solve Ku = f
    U[:, j] = u                         # filling in values of u for m samples...each column corresponds to a different sample

noise = 0 * np.random.randn(*U.shape)
U_noisy = U * (1 + noise)

# mean + centre the data
U_mean = np.mean(U, axis=1, keepdims=True)
U_centered = U_noisy - U_mean

#covariance matrix
cov_matrix = (1/m) * (U_centered @ U_centered.T)

#eigenvalues of covariance matrix
evalues_cm, evectors_cm = np.linalg.eig(cov_matrix) 

# sort in descending order
idx = np.argsort(evalues_cm)[::1]
evalues = evalues_cm[idx] #reordering rows
evectors = evectors_cm[:, idx] #reordering columns

error_fraction = 0.1

#since the covariance matrix is symmetric, the eigenvectors are already perpendicular to eachother
#to calculate optimum dimension r by checking error fraction:
evalues_sum = np.sum(evalues)
frac = 1
r = 0

while frac > error_fraction:
    error_sum = np.sum(evalues[:(n-r)])
    frac = error_sum/evalues_sum
    r = r+1

#calculating MSE vs r
errors = []

for r in range(1, n+1):
    Vr = evectors[:, :r]
    
    U_proj = Vr @ (Vr.T @ U_centered) #calculating u in direction of eigen vectors
    error = np.mean((U_centered - U_proj)**2)
    
    errors.append(error)

#to plot mean square error v/s r
plt.plot(range(1, n+1), errors)
plt.xlabel("Reduced dimension r")
plt.ylabel("Mean Square Error")
plt.title("PCA Error vs Dimension")
plt.show()

print("Original Dimension: ", n)
print("reduced dimension is:", r)
print("Principal directions (PCA):", evectors[:, :r])
     


### Comparing PCA and Mode Shape Approach

- Compare by computing |dot product| between corresponding vectors.
- If they are aligned, |dot| ~ 1. If orthogonal, |dot| ~ 0.
- eigenvectors can be sign-flipped, so we use absolute value.

In [ ]:
num_compare = min(10, n)   # compare first 10 directions
dot_products = []

for i in range(num_compare):
    mode_vec = evec_K[:, i]           # i-th mode shape (sorted by ascending eigenvalue)
    pca_vec  = evectors[:, i]  # i-th PCA direction (sorted by descending variance)
    dot = abs(np.dot(mode_vec, pca_vec))
    dot_products.append(dot)

plt.figure()
plt.bar(range(1, num_compare+1), dot_products)
plt.xlabel("Direction index")
plt.ylabel("|cos θ| between Mode Shape and PCA vectors")
plt.title("Alignment of Mode Shape vs PCA Principal Directions")
plt.ylim([0, 1])
plt.axhline(y=0.9, color='r', linestyle='--', label='0.9 threshold')
plt.legend()
plt.grid(True)
plt.show()

print("Dot products (|cos θ|) between mode shape and PCA directions:")
for i, d in enumerate(dot_products):
    print(f"  Direction {i+1}: {d:.4f}  {'<-- nearly aligned' if d > 0.9 else '<-- NOT aligned'}")